In [ ]:
import os
import numpy as np
import pandas as pd

# ============ 0. CONFIGURACIÓN ============
RUTA_SDEM = "/Users/claudioalejoencarnacionmartinez/Documents/SEMESTRE OTOÑO 2026/LAB APRENDIZAJE ESTADÍSTICO/LABORATORIO-DE-APRENDIZAJE-ESTAD-STICO/SEGUNDO PARCIAL/PROYECTO_02/conjunto_de_datos_sdem_enoe_2026_2t.csv"
CARPETA_SALIDA = "/Users/claudioalejoencarnacionmartinez/Documents/SEMESTRE OTOÑO 2026/LAB APRENDIZAJE ESTADÍSTICO/LABORATORIO-DE-APRENDIZAJE-ESTAD-STICO/SEGUNDO PARCIAL/PROYECTO_02"

NUMERICAS   = ["eda", "anios_esc", "hrsocup", "log_ing"]
BINARIAS    = ["hrs_cero", "ing_cero"]
CATEGORICAS = ["sex", "e_con", "niv_ins", "cs_p12", "cs_p17",
               "t_loc_tri", "zona", "ur", "rama", "scian", "dur9c"]
PREDICTORAS = NUMERICAS + BINARIAS + CATEGORICAS

CODIGO_INFORMAL = 1          # <- PENDIENTE: confirma en el catálogo de emp_ppal
MUNICIPIOS_AMG5 = [39, 120, 98, 101, 97]   # GDL, Zapopan, Tlaquepaque, Tonalá, Tlajomulco
ELIMINAR_NULOS  = True       # False -> deja los NaN para imputarlos dentro del Pipeline

# Códigos de "no especificado" -> NaN. VERIFICA cada uno con el diccionario y la tabla de reemplazos
INVALIDOS = {
    "anios_esc": [99],
    "e_con":     [9],
    "cs_p12":    [9],
    "cs_p17":    [9],
}

# ============ 1. CARGA ============
usar = set(ID_COLS + FILTRO_COLS + PREDICTORAS)
raw = pd.read_csv(RUTA_SDEM, dtype=str, encoding="latin-1",                  # si falla: encoding="latin-1"
                  usecols=lambda c: c.strip().lower() in usar)
raw.columns = raw.columns.str.strip().str.lower()
raw = raw.apply(lambda s: s.str.strip())
print(f"Datos originales: {raw.shape}")

# ============ 2. TIPOS ============
for c in ID_COLS + FILTRO_COLS + PREDICTORAS:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")

# ============ 3. FILTROS ESTRUCTURALES ============
# Base 100% = entrevistas completas
df = raw[raw["r_def"] == 0]
BASE = len(df)
resumen = [("Total en la base (todas las personas)", len(raw)),
           ("Entrevista completa (r_def = 0)  [BASE 100%]", BASE)]

pasos = [
    ("Población ocupada (clase2 = 1)",        lambda d: d["clase2"] == 1),
    ("Edad >= 15 (excluye 99 = no especif.)", lambda d: (d["eda"] >= 15) & (d["eda"] != 99)),
    ("Etiqueta válida (emp_ppal en {1, 2})",  lambda d: d["emp_ppal"].isin([1, 2])),
]
for nombre, regla in pasos:
    df = df[regla(df)]
    resumen.append((nombre, len(df)))
df = df.copy()

# ============ 4. LIMPIEZA DE CÓDIGOS ESPECIALES ============
reemp = []
for col, codigos in INVALIDOS.items():
    mask = df[col].isin(codigos)
    reemp.append((col, codigos, int(mask.sum()), round(100 * mask.sum() / BASE, 2)))
    df.loc[mask, col] = np.nan
print("\nCódigos especiales reemplazados por NaN:")
print(pd.DataFrame(reemp, columns=["variable", "códigos", "filas", "% de la base"]).to_string(index=False))

# ============ 5. NULOS ============
nulos = df[PREDICTORAS].isna().sum()
print("\nNulos por predictora (después de filtros y limpieza):")
print(pd.DataFrame({"nulos": nulos, "% de la base": (100 * nulos / BASE).round(2)})
        .query("nulos > 0").to_string() if nulos.sum() else "ninguno")

if ELIMINAR_NULOS:
    antes = len(df)
    df = df.dropna(subset=PREDICTORAS)
    resumen.append(("Sin nulos en predictoras", len(df)))
    print(f"\nFilas eliminadas por nulos: {antes - len(df)} "
          f"({100 * (antes - len(df)) / BASE:.2f}% de la base de entrevistas completas)")

tabla = pd.DataFrame(resumen, columns=["Paso", "Filas"])
tabla["Eliminadas"] = (-tabla["Filas"].diff()).fillna(0).astype(int)
tabla["% de la base"] = (100 * tabla["Filas"] / BASE).round(2)
tabla.loc[0, ["Eliminadas", "% de la base"]] = [0, np.nan]        # fila inicial no es parte de la base
print("\n", tabla.to_string(index=False))
perdida_total = 100 * (BASE - len(df)) / BASE
print(f"\nPérdida total sobre entrevistas completas: {perdida_total:.2f}% "
      "(incluye a quienes no son ocupados, menores de 15, etc.)")

df["hrs_cero"] = (df["hrsocup"] == 0).astype(int)
df["ing_cero"] = (df["ingocup"] == 0).astype(int)
df["log_ing"]  = np.log1p(df["ingocup"])

# ============ 6. ETIQUETA Y FORMATO FINAL ============
df["y"] = (df["emp_ppal"] == CODIGO_INFORMAL).astype(int)
df = df.drop(columns=["emp_ppal", "r_def", "clase2"])

for c in ["cve_ent", "cve_mun"]:
    df[c] = df[c].astype("Int64")
for c in CATEGORICAS:
    df[c] = df[c].astype("Int64").astype("string")
df = df.reset_index(drop=True)

# ============ 7. LOS TRES DATAFRAMES ============
df_nacional = df.copy()
df_jalisco  = df[df["cve_ent"] == 14].copy()
df_amg5     = df_jalisco[df_jalisco["cve_mun"].isin(MUNICIPIOS_AMG5)].copy()

assert len(df_amg5) <= len(df_jalisco) <= len(df_nacional)
if ELIMINAR_NULOS:
    for nom, d in [("AMG5", df_amg5), ("Jalisco", df_jalisco), ("Nacional", df_nacional)]:
        assert d[PREDICTORAS].isna().sum().sum() == 0, f"Hay nulos en {nom}"

print("\nResumen de los tres dataframes:")
print(pd.DataFrame({
    "filas": [len(df_amg5), len(df_jalisco), len(df_nacional)],
    "% informal (y=1)": [round(100 * d["y"].mean(), 1) for d in (df_amg5, df_jalisco, df_nacional)],
}, index=["AMG5", "Jalisco", "Nacional"]))

# ============ 8. GUARDADO ============
os.makedirs(CARPETA_SALIDA, exist_ok=True)
for nombre, d in [("amg5", df_amg5), ("jalisco", df_jalisco), ("nacional", df_nacional)]:
    ruta = f"{CARPETA_SALIDA}/enoe_{nombre}_limpio.csv"
    d.to_csv(ruta, index=False)
    mb = os.path.getsize(ruta) / 1e6
    print(f"{ruta}: {mb:.1f} MB" + ("  <- GitHub avisa desde 50 MB y rechaza desde 100 MB" if mb > 50 else ""))

Datos originales: (417128, 21)

Códigos especiales reemplazados por NaN:
 variable  códigos  filas  % de la base
anios_esc     [99]    622          0.15
  hrsocup      [0]   7257          1.74
  ingocup [999998]      0          0.00
    e_con      [9]     28          0.01
   cs_p12      [9]     18          0.00
   cs_p17      [9]     22          0.01

Nulos por predictora (después de filtros y limpieza):
           nulos  % de la base
anios_esc    622          0.15
hrsocup     7257          1.74
e_con         28          0.01
cs_p12        18          0.00
cs_p17        22          0.01

Filas eliminadas por nulos: 7895 (1.90% de la base de entrevistas completas)

                                         Paso  Filas  Eliminadas  % de la base
       Total en la base (todas las personas) 417128           0           NaN
Entrevista completa (r_def = 0)  [BASE 100%] 416393         735        100.00
              Población ocupada (clase2 = 1) 190325      226068         45.71
       Edad >=

In [18]:
import os
import numpy as np
import pandas as pd

# ============ 0. CONFIGURACIÓN ============
RUTA_SDEM = "/Users/claudioalejoencarnacionmartinez/Documents/SEMESTRE OTOÑO 2026/LAB APRENDIZAJE ESTADÍSTICO/LABORATORIO-DE-APRENDIZAJE-ESTAD-STICO/SEGUNDO PARCIAL/PROYECTO_02/conjunto_de_datos_sdem_enoe_2026_2t.csv"
CARPETA_SALIDA = "/Users/claudioalejoencarnacionmartinez/Documents/SEMESTRE OTOÑO 2026/LAB APRENDIZAJE ESTADÍSTICO/LABORATORIO-DE-APRENDIZAJE-ESTAD-STICO/SEGUNDO PARCIAL/PROYECTO_02"

ID_COLS     = ["cve_ent", "cve_mun", "fac_tri"]   # trazabilidad, NO son predictoras
FILTRO_COLS = ["r_def", "clase2", "emp_ppal"]
DIAG_COLS   = ["ing7c"]                           # solo para diagnóstico, se descarta al final

BRUTAS_NUM  = ["eda", "anios_esc", "hrsocup", "ingocup"]
NUMERICAS   = ["eda", "anios_esc", "hrsocup", "log_ing"]
BINARIAS    = ["hrs_cero", "ing_cero"]
CATEGORICAS = ["sex", "e_con", "niv_ins", "cs_p12", "cs_p17",
               "t_loc_tri", "zona", "ur", "rama", "scian", "dur9c"]
PREDICTORAS = NUMERICAS + BINARIAS + CATEGORICAS

CODIGO_INFORMAL = 1          # <- PENDIENTE: confirma en el catálogo de emp_ppal
MUNICIPIOS_AMG5 = [39, 120, 98, 101, 97]   # GDL, Zapopan, Tlaquepaque, Tonalá, Tlajomulco
ELIMINAR_NULOS  = True       # False -> deja los NaN para imputarlos dentro del Pipeline

# Códigos de "no especificado" -> NaN
# hrsocup = 0 se conserva (ocupados que no trabajaron la semana de referencia)
# ingocup = 0 se conserva (sin ingreso); no hay código 999998 en esta edición
INVALIDOS = {
    "anios_esc": [99],
    "e_con":     [9],
    "cs_p12":    [9],
    "cs_p17":    [9],
}

# ============ 1. CARGA ============
usar = set(ID_COLS + FILTRO_COLS + BRUTAS_NUM + CATEGORICAS + DIAG_COLS)
raw = pd.read_csv(RUTA_SDEM, dtype=str, encoding="latin-1", usecols=lambda c: c.strip().lower() in usar)
raw.columns = raw.columns.str.strip().str.lower()
raw = raw.apply(lambda s: s.str.strip())
print(f"Datos originales: {raw.shape}")

# ============ 2. TIPOS ============
for c in ID_COLS + FILTRO_COLS + BRUTAS_NUM + CATEGORICAS + DIAG_COLS:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")

# ============ 3. FILTROS ESTRUCTURALES ============
df = raw[raw["r_def"] == 0]
BASE = len(df)
resumen = [("Total en la base (todas las personas)", len(raw)),
           ("Entrevista completa (r_def = 0)  [BASE 100%]", BASE)]

pasos = [
    ("Población ocupada (clase2 = 1)",        lambda d: d["clase2"] == 1),
    ("Edad >= 15 (excluye 99 = no especif.)", lambda d: (d["eda"] >= 15) & (d["eda"] != 99)),
    ("Etiqueta válida (emp_ppal en {1, 2})",  lambda d: d["emp_ppal"].isin([1, 2])),
]
for nombre, regla in pasos:
    df = df[regla(df)]
    resumen.append((nombre, len(df)))
df = df.copy()

# ============ 3b. DIAGNÓSTICOS (antes de transformar) ============
print("\n--- Diagnóstico: ingocup == 0 contra ing7c (revisa el catálogo de ing7c) ---")
print(pd.crosstab(df["ingocup"] == 0, df["ing7c"], margins=True))
print("\nTasa de informalidad con ingocup == 0 :",
      round((df.loc[df["ingocup"] == 0, "emp_ppal"] == CODIGO_INFORMAL).mean(), 3))
print("Tasa de informalidad con ingocup  > 0 :",
      round((df.loc[df["ingocup"] > 0, "emp_ppal"] == CODIGO_INFORMAL).mean(), 3))
print("\nhrsocup > 100:", (df["hrsocup"] > 100).sum(), " | hrsocup > 120:", (df["hrsocup"] > 120).sum())

# ============ 4. LIMPIEZA DE CÓDIGOS ESPECIALES ============
reemp = []
for col, codigos in INVALIDOS.items():
    mask = df[col].isin(codigos)
    reemp.append((col, codigos, int(mask.sum()), round(100 * mask.sum() / BASE, 2)))
    df.loc[mask, col] = np.nan
print("\nCódigos especiales reemplazados por NaN:")
print(pd.DataFrame(reemp, columns=["variable", "códigos", "filas", "% de la base"]).to_string(index=False))

# ============ 4b. VARIABLES DERIVADAS (determinísticas, sin fuga) ============
df["hrs_cero"] = (df["hrsocup"] == 0).astype(int)
df["ing_cero"] = (df["ingocup"] == 0).astype(int)
df["log_ing"]  = np.log1p(df["ingocup"])

# ============ 5. NULOS ============
nulos = df[PREDICTORAS].isna().sum()
if nulos.sum():
    print("\nNulos por predictora:")
    print(pd.DataFrame({"nulos": nulos, "% de la base": (100 * nulos / BASE).round(2)})
            .query("nulos > 0").to_string())
else:
    print("\nNulos por predictora: ninguno")

if ELIMINAR_NULOS:
    antes = len(df)
    df = df.dropna(subset=PREDICTORAS)
    resumen.append(("Sin nulos en predictoras", len(df)))
    print(f"\nFilas eliminadas por nulos: {antes - len(df)} "
          f"({100 * (antes - len(df)) / BASE:.2f}% de la base de entrevistas completas)")

tabla = pd.DataFrame(resumen, columns=["Paso", "Filas"])
tabla["Eliminadas"] = (-tabla["Filas"].diff()).fillna(0).astype(int)
tabla["% de la base"] = (100 * tabla["Filas"] / BASE).round(2)
tabla.loc[0, ["Eliminadas", "% de la base"]] = [0, np.nan]
print("\n", tabla.to_string(index=False))

# ============ 6. ETIQUETA Y FORMATO FINAL ============
df["y"] = (df["emp_ppal"] == CODIGO_INFORMAL).astype(int)
df = df.drop(columns=["emp_ppal", "r_def", "clase2", "ing7c"])

for c in ["cve_ent", "cve_mun"]:
    df[c] = df[c].astype("Int64")
for c in CATEGORICAS:
    df[c] = df[c].astype("Int64").astype("string")

# ingocup se conserva solo como referencia; NO es predictora (se usa log_ing)
COLS_FINALES = ID_COLS + ["ingocup"] + PREDICTORAS + ["y"]
df = df[COLS_FINALES].reset_index(drop=True)

# ============ 7. LOS TRES DATAFRAMES ============
df_nacional = df.copy()
df_jalisco  = df[df["cve_ent"] == 14].copy()
df_amg5     = df_jalisco[df_jalisco["cve_mun"].isin(MUNICIPIOS_AMG5)].copy()

assert len(df_amg5) <= len(df_jalisco) <= len(df_nacional)
assert "ingocup" not in PREDICTORAS
if ELIMINAR_NULOS:
    for nom, d in [("AMG5", df_amg5), ("Jalisco", df_jalisco), ("Nacional", df_nacional)]:
        assert d[PREDICTORAS].isna().sum().sum() == 0, f"Hay nulos en {nom}"

print("\nResumen de los tres dataframes:")
print(pd.DataFrame({
    "filas": [len(df_amg5), len(df_jalisco), len(df_nacional)],
    "% informal (y=1)": [round(100 * d["y"].mean(), 1) for d in (df_amg5, df_jalisco, df_nacional)],
}, index=["AMG5", "Jalisco", "Nacional"]))

# ============ 8. GUARDADO (sobrescribe los .csv anteriores) ============
os.makedirs(CARPETA_SALIDA, exist_ok=True)
for nombre, d in [("amg5", df_amg5), ("jalisco", df_jalisco), ("nacional", df_nacional)]:
    ruta = f"{CARPETA_SALIDA}/enoe_{nombre}_limpio.csv"
    d.to_csv(ruta, index=False)
    print(f"{ruta}: {os.path.getsize(ruta) / 1e6:.1f} MB")

Datos originales: (417128, 22)

--- Diagnóstico: ingocup == 0 contra ing7c (revisa el catálogo de ing7c) ---
ing7c        1      2      3     4     5     6      7     All
ingocup                                                      
False    63212  47427   8237  3638  1032     0      0  123546
True     21465  13887   3003   711   204  7440  19097   65807
All      84677  61314  11240  4349  1236  7440  19097  189353

Tasa de informalidad con ingocup == 0 : 0.479
Tasa de informalidad con ingocup  > 0 : 0.541

hrsocup > 100: 275  | hrsocup > 120: 22

Códigos especiales reemplazados por NaN:
 variable códigos  filas  % de la base
anios_esc    [99]    622          0.15
    e_con     [9]     28          0.01
   cs_p12     [9]     18          0.00
   cs_p17     [9]     22          0.01

Nulos por predictora:
           nulos  % de la base
anios_esc    622          0.15
e_con         28          0.01
cs_p12        18          0.00
cs_p17        22          0.01

Filas eliminadas por nulos: 680